# فصل ۷: ساخت برنامه‌های چت
## شروع سریع API OpenAI

این دفترچه از [مخزن نمونه‌های Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) اقتباس شده است که شامل دفترچه‌های دسترسی به خدمات [Azure OpenAI](notebook-azure-openai.ipynb) می‌باشد.

API پایتون OpenAI همچنین با مدل‌های Azure OpenAI کار می‌کند، با چند تغییر کوچک. درباره تفاوت‌ها بیشتر بدانید: [چگونگی تغییر بین نقطه پایان OpenAI و Azure OpenAI با پایتون](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# مرور کلی  
«مدل‌های زبانی بزرگ تابع‌هایی هستند که متن را به متن نگاشت می‌کنند. با دادن یک رشته ورودی متنی، یک مدل زبانی بزرگ سعی می‌کند متنی که در ادامه خواهد آمد را پیش‌بینی کند»(1). این دفترچه یادداشت «شروع سریع» مفاهیم سطح بالا درباره مدل‌های زبان بزرگ را به کاربران معرفی می‌کند، ملزومات اصلی بسته برای شروع کار با AML، معرفی نرم به طراحی پرسش و چندین مثال کوتاه از کاربردهای مختلف را ارائه می‌دهد.


## فهرست مطالب  

[بررسی اجمالی](#overview)  
[نحوه استفاده از سرویس OpenAI](#how-to-use-openai-service)  
[1. ایجاد سرویس OpenAI خود](#1.-creating-your-openai-service)  
[2. نصب](#2.-installation)    
[3. اطلاعات ورود](#3.-credentials)  

[موارد استفاده](#use-cases)    
[1. خلاصه‌سازی متن](#1.-summarize-text)  
[2. دسته‌بندی متن](#2.-classify-text)  
[3. تولید نام‌های جدید محصول](#3.-generate-new-product-names)  
[4. تنظیم دقیق یک دسته‌بند](#4.fine-tune-a-classifier)  

[مراجع](#references)


### اولین پرامپت خود را بسازید  
این تمرین کوتاه مقدمه‌ای پایه برای ارسال پرامپت به مدل OpenAI برای یک کار ساده «خلاصه‌سازی» فراهم می‌کند.


**مراحل**:  
1. کتابخانه OpenAI را در محیط پایتون خود نصب کنید  
2. کتابخانه‌های کمکی استاندارد را بارگذاری کنید و اعتبارهای امنیتی معمول OpenAI خود را برای سرویس OpenAI که ایجاد کرده‌اید تنظیم کنید  
3. مدلی برای کار خود انتخاب کنید  
4. یک پرامپت ساده برای مدل ایجاد کنید  
5. درخواست خود را به API مدل ارسال کنید!


### ۱. نصب OpenAI


In [ ]:
%pip install openai python-dotenv

### ۲. وارد کردن کتابخانه‌های کمکی و نمونه‌سازی اعتبارنامه‌ها


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### ۳. یافتن مدل مناسب  
مدل‌هایی مانند GPT-4o و GPT-4o mini قادر به درک و تولید زبان طبیعی هستند و در پلتفرم OpenAI با سطوح مختلفی از قدرت و سرعت که برای وظایف گوناگون مناسب است، در دسترس می‌باشند.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## ۴. طراحی درخواست  

«جادوی مدل‌های زبان بزرگ این است که با آموزش برای کمینه کردن خطای پیش‌بینی در حجم وسیعی از متن‌ها، مدل‌ها در نهایت مفاهیمی را یاد می‌گیرند که برای این پیش‌بینی‌ها مفید هستند. برای مثال، آن‌ها مفاهیمی مثل»(1):

* چگونه املا کنند
* چگونه دستور زبان کار می‌کند
* چگونه پارافرایز کنند
* چگونه به سوالات پاسخ دهند
* چگونه یک مکالمه را پیش ببرند
* چگونه به زبان‌های مختلف بنویسند
* چگونه برنامه‌نویسی کنند
* و غیره.

#### چگونه یک مدل زبان بزرگ را کنترل کنیم  
«از میان همه ورودی‌ها به یک مدل زبان بزرگ، بدون شک موثرترین آن‌ها متن درخواست است(1).

مدل‌های زبان بزرگ را می‌توان به چند روش به تولید خروجی وادار کرد:

دستورالعمل: به مدل بگویید چه چیزی می‌خواهید
تکمیل: مدل را وادار کنید ابتدای چیزی را که می‌خواهید کامل کند
نمایش: به مدل نشان دهید چه چیزی می‌خواهید، با یکی از روش‌های زیر:
چند نمونه در درخواست
صدها یا هزاران نمونه در یک مجموعه داده آموزش تنظیم دقیق»



#### سه راهنمایی اساسی برای ساخت درخواست‌ها وجود دارد:

**نشان بدهید و توضیح دهید**. مشخص کنید چه می‌خواهید، چه از طریق دستورالعمل، چه مثال، یا ترکیبی از هر دو. اگر می‌خواهید مدل یک لیست از آیتم‌ها را به ترتیب الفبایی مرتب کند یا یک پاراگراف را بر اساس احساس دسته‌بندی کند، نشان دهید که این همان چیزی است که می‌خواهید.

**داده‌های با کیفیت ارائه دهید**. اگر سعی دارید یک دسته‌بند بسازید یا مدل را به دنبال کردن الگویی وادار کنید، مطمئن شوید نمونه‌های کافی وجود دارد. حتماً نمونه‌هایتان را بازبینی کنید — مدل معمولاً به اندازه کافی باهوش است که اشتباهات املایی پایه را تشخیص دهد و پاسخ دهد، اما ممکن است فرض کند این کار عمدی است و این می‌تواند روی پاسخ تاثیر بگذارد.

**تنظیمات خود را بررسی کنید.** دما (temperature) و top_p کنترل می‌کنند که مدل در تولید پاسخ چقدر قطعی باشد. اگر پاسخ جایی دارد که فقط یک جواب درست وجود دارد، پس بهتر است این‌ها را پایین تنظیم کنید. اگر به دنبال پاسخ‌های متنوع‌تر هستید، ممکن است بخواهید آن‌ها را بالاتر تنظیم کنید. اشتباه شماره یک مردم در این تنظیمات این است که فکر می‌کنند این‌ها کنترل‌های «هوشمندی» یا «خلاقیت» هستند.


منبع: https://learn.microsoft.com/azure/ai-services/openai/overview


### ۵. ارسال کنید!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### همان فراخوانی را تکرار کنید، نتایج چگونه مقایسه می‌شوند؟


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## خلاصه کردن متن  
#### چالش  
متن را با افزودن یک 'tl;dr:' در پایان یک بخش متنی خلاصه کنید. توجه کنید که مدل چگونه بدون دستورالعمل‌های اضافی می‌داند چگونه چندین کار مختلف را انجام دهد. شما می‌توانید با استفاده از پرسش‌های توصیفی‌تر از tl;dr آزمایش کنید تا رفتار مدل را تغییر داده و خلاصه‌سازی دریافتی خود را سفارشی کنید(3).  

کارهای اخیر نشان داده‌اند که پیش‌آموزش روی یک مجموعه بزرگ از متن و سپس انجام تنظیم دقیق روی یک وظیفه خاص، بهبودهای قابل توجهی در بسیاری از وظایف و معیارهای پردازش زبان طبیعی به همراه داشته است. اگرچه این روش معماری مستقل از وظیفه دارد، اما همچنان به مجموعه‌داده‌های تنظیم دقیق وظیفه‌محور با هزاران یا ده‌ها هزار مثال نیاز دارد. در مقابل، انسان‌ها به‌طور کلی می‌توانند یک وظیفه زبانی جدید را تنها با چند مثال یا دستورالعمل ساده انجام دهند - چیزی که سیستم‌های فعلی پردازش زبان طبیعی هنوز تا حد زیادی در انجام آن مشکل دارند. در اینجا نشان می‌دهیم که افزایش مقیاس مدل‌های زبانی عملکرد چندنمونه‌ای مستقل از وظیفه را به طور چشمگیری بهبود می‌بخشد و گاهی حتی به رقابت با روش‌های تنظیم دقیق پیشرفته قبلی می‌رسد.  



خلاصه  


# تمرین‌ها برای چندین مورد استفاده  
1. خلاصه کردن متن  
2. طبقه‌بندی متن  
3. تولید نام‌های جدید محصول  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## دسته‌بندی متن  
#### چالش  
موارد را به دسته‌هایی که در زمان استنتاج ارائه می‌شوند، دسته‌بندی کنید. در مثال زیر، هم دسته‌ها و هم متنی که باید دسته‌بندی شود را در پرامپت (*playground_reference) ارائه می‌دهیم. 

سوال مشتری: سلام، یکی از کلیدهای کیبورد لپ‌تاپ من اخیراً شکسته و به یک جایگزین نیاز دارم:

دسته‌بندی شده: 


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## تولید نام‌های جدید محصول
#### چالش
نام‌های محصول را از کلمات نمونه ایجاد کنید. در اینجا اطلاعاتی درباره محصولی که قرار است برای آن نام تولید کنیم در پرامپت وارد کرده‌ایم. همچنین یک نمونه مشابه را ارائه داده‌ایم تا الگوی مورد نظر را نشان دهیم. مقدار دما را نیز بالا تنظیم کرده‌ایم تا تصادفی بودن و پاسخ‌های نوآورانه افزایش یابد.

توضیحات محصول: دستگاه ساخت میلک‌شیک خانگی
کلمات پایه: سریع، سالم، جمع‌وجور.
نام‌های محصول: HomeShaker, Fit Shaker, QuickShake, Shake Maker

توضیحات محصول: یک جفت کفش که می‌تواند برای هر سایز پا مناسب باشد.
کلمات پایه: سازگار، مناسب، همه‌سایز.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# منابع  
- [کتابچه راهنمای Openai](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [پرتال Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [بهترین روش‌ها برای تنظیم دقیق مدل‌های GPT جهت طبقه‌بندی متن](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# برای دریافت کمک بیشتر  
[تیم تجاری‌سازی OpenAI](AzureOpenAITeam@microsoft.com) 


# مشارکت‌کنندگان
* لوئیس لی  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
